In [1]:
import hopsworks
import os
import dotenv

dotenv.load_dotenv()

# Load Feature Group
feature_group_name = "weather_prediction"
features = ['temperature_2m', 'relative_humidity_2m', 'precipitation',
       'wind_speed_10m', 'cloud_cover', 'surface_pressure', 'dew_point_2m', 'pressure_msl',
       'derived_hour', 'derived_month', 'relative_humidity_2m_24h_avg', 'pressure_msl_3h_pct_change']
labels = ["precipation_label"]

project = hopsworks.login(
    project='mlops',  # Replace with your project name
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=os.environ["HOPSWORKS_API_KEY"]  # Get from Hopsworks UI > Account Settings > API Keys
)

fs = project.get_feature_store()
# Get all versions of the feature group
fgs = fs.get_feature_groups()
fg_versions = [fg.version for fg in fgs if fg.name == feature_group_name]

# Load latest version of feature group as DataFrame
fg = fs.get_feature_group(name=feature_group_name, version=max(fg_versions))
df = fg.read()

print(f"Loaded {feature_group_name} with version {max(fg_versions)}")
print(f"Features and Labels:\n{fg.features}")
print(df.columns)

/home/tobias/programmieren/aiops_mlops_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-04-09 19:40:55,616 INFO: Initializing external client
2026-04-09 19:40:55,616 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-04-09 19:40:56,969 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31926
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.38s) 
Loaded weather_prediction with version 2
Features and Labels:
[Feature('partition', 'string', None, False, False, True, 'varchar(100)', None, 37866), Feature('temperature_2m', 'float', None, False, False, False, 'float', None, 37866), Feature('relative_humidity_2m', 'float', None, False, False, False, 'float', None, 37866), Feature('precipitation', 'float', None, False, False, False, 'float', None, 37866), Feature('precipitation_probability', 'float', None, False, False, False, 'float', None, 37866), Feature('weather_code', 'float', None, False, False, False, 'float', None, 37866), Feature('wind_speed_10m', 'float', None, Fal

In [3]:
# Create Feature View with features and labels

query = fg.select(features+labels)

feature_view = fs.create_feature_view(
    name="weather_prediction",
    version=fg.version, # Always use the same version number als the referenced feature group
    query=query,
    labels=labels,
    description="Feature View for weather prediction"
)


RestAPIError: Metadata operation error: (url: https://eu-west.cloud.hopsworks.ai/hopsworks-api/api/project/31926/featurestores/20610/featureview). Server response: 
HTTP code: 400, HTTP reason: Bad Request, body: b'{"errorCode":270179,"usrMsg":"Feature view: weather_prediction, version: 2","errorMsg":"The provided feature view name and version already exists"}', error code: 270179, error msg: The provided feature view name and version already exists, user msg: Feature view: weather_prediction, version: 2

In [6]:
# Create the training and test datasets
# Random DataSets need to be created outside of Hopsworks
from sklearn.model_selection import train_test_split
import pandas as pd


# Separate features and labels
X = pd.DataFrame(index=df.index)
y = pd.DataFrame(index=df.index)

for feature in features:
    X[feature] = df[feature]
print(X.info())

for label in labels:
    y[label] = df[label]
# If y has only one column, convert it to a Series
if len(y.columns) == 1:
    y = y.iloc[:, 0]  # or y[y.columns[0]]
print(y.info())

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    test_size=0.2,
    random_state=42,  # For reproducibility
)
query = fg.select(features + labels)
fv = fs.get_feature_view("weather_prediction", version=fg.version)

# Upload Dataset to hopsworks
train_dataset = fs.create_training_dataset(
    name="weather_prediction_train",
    version=fg.version,
    description="Training Dataset for weather_prediction. Version number is the same as feature group",
    data_format="tfrecords",
    coalesce=False,
    storage_connector=None,
    splits={"train": 0.8, "test": 0.2},
    location="",
    seed=42,
    statistics_config=None,
    label=labels,
    transformation_functions={},
    train_split="train",
)

train_dataset.save(query)





<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1555 entries, 0 to 1554
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   temperature_2m                1555 non-null   float32
 1   relative_humidity_2m          1555 non-null   float32
 2   precipitation                 1555 non-null   float32
 3   wind_speed_10m                1555 non-null   float32
 4   cloud_cover                   1555 non-null   float32
 5   surface_pressure              1555 non-null   float32
 6   dew_point_2m                  1555 non-null   float32
 7   pressure_msl                  1555 non-null   float32
 8   derived_hour                  1555 non-null   int64  
 9   derived_month                 1555 non-null   int64  
 10  relative_humidity_2m_24h_avg  1555 non-null   float64
 11  pressure_msl_3h_pct_change    1555 non-null   float32
dtypes: float32(9), float64(1), int64(2)
memory usage: 91.2 KB
None

RestAPIError: Metadata operation error: (url: https://eu-west.cloud.hopsworks.ai/hopsworks-api/api/project/31926/featurestores/20610/trainingdatasets). Server response: 
HTTP code: 400, HTTP reason: Bad Request, body: b'{"errorCode":270016,"usrMsg":"Training Dataset: weather_prediction_train, version: 2","errorMsg":"The provided training dataset name already exists"}', error code: 270016, error msg: The provided training dataset name already exists, user msg: Training Dataset: weather_prediction_train, version: 2

In [ ]:
# Train and evaluate model with the local split
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=300, random_state=42)
# from xgboost import XGBRegressor
# model = XGBRegressor(objective='reg:squarederror', n_estimators=300, random_state=42)
model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)
y_pred = model.predict(X_test)
print(accuracy) # Low Value should be ok since we return a continious number

0.2886031739333904


In [ ]:
# Save model locally
import joblib
from pathlib import Path
modelpath = Path("tmp")/f"weather_prediction_model_{fg.version}.joblib"
joblib.dump(model, modelpath)

# Save model to hopsworks




['tmp/weather_prediction_model_2.joblib']

In [ ]:
# create a training dataset 
X_train, y_train, X_test, y_test = fv.train_test_split(test_size=0.2, description="Automatic creation")


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.37s) 
2026-04-09 19:51:16,944 INFO: Computing insert statistics
2026-04-09 19:51:16,958 INFO: Computing insert statistics
2026-04-09 19:51:18,023 INFO: Provenance cached data - overwriting last accessed/created training dataset from 9 to 10.


In [66]:
# Training Datasets for the feature view increment each run. They are bound to the feature Group and feature view. 
# So we just get the newest one insteag of the feature group version. 

training_dataset_versions = fv.get_training_datasets()
training_dataset_max_version = max([x.version for x in training_dataset_versions])
# Documentation seems to be wrong https://docs.hopsworks.ai/3.0/user_guides/fs/feature_view/training-data/#read-training-data
X_train, X_test, y_train, y_test = fv.get_train_test_split(training_dataset_version=training_dataset_max_version)
X_train.drop(columns=["timestamp"], axis=1, inplace=True)
X_test.drop(columns=["timestamp"], axis=1, inplace=True)

# Normalization of features and labels
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train[features] = scaler.fit_transform(X_train[features])
X_test[features] = scaler.fit_transform(X_test[features])
y_train[labels] = scaler.fit_transform(y_train[labels])
y_test[labels] = scaler.fit_transform(y_test[labels])

# for col in features:
#     X_train[col] = (X_train[col] - X_train[col].min()) / (X_train[col].max() - X_train[col].min())
#     X_test[col] = (X_test[col] - X_test[col].min()) / (X_test[col].max() - X_test[col].min())

# for col in labels:
#     y_train[col] = (y_train[col] - y_train[col].min()) / (y_train[col].max() - y_train[col].min())
#     y_test[col] = (y_test[col] - y_test[col].min()) / (y_test[col].max() - y_test[col].min())


print("X_train:", len(X_train),"X_test", len(X_test), "y_train", len(y_train), "y_test", len(y_test))



Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.38s) 
2026-04-09 20:31:43,853 INFO: Computing insert statistics
2026-04-09 20:31:43,864 INFO: Computing insert statistics
2026-04-09 20:31:45,066 INFO: Provenance cached data - overwriting last accessed/created training dataset from 10 to 10.
X_train: 1244 X_test 311 y_train 1244 y_test 311


In [67]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1244 entries, 0 to 1554
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   temperature_2m                1244 non-null   float64
 1   relative_humidity_2m          1244 non-null   float64
 2   precipitation                 1244 non-null   float64
 3   wind_speed_10m                1244 non-null   float64
 4   cloud_cover                   1244 non-null   float64
 5   surface_pressure              1244 non-null   float64
 6   dew_point_2m                  1244 non-null   float64
 7   pressure_msl                  1244 non-null   float64
 8   derived_hour                  1244 non-null   float64
 9   derived_month                 1244 non-null   float64
 10  relative_humidity_2m_24h_avg  1244 non-null   float64
 11  pressure_msl_3h_pct_change    1244 non-null   float64
dtypes: float64(12)
memory usage: 126.3 KB


In [ ]:
# Train the model with the Dataset from the 
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=300, random_state=42)
# from xgboost import XGBRegressor
# model = XGBRegressor(objective='reg:squarederror', n_estimators=300, random_state=42)
model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)
y_pred = model.predict(X_test)
print(accuracy) # Low Value should be ok since we return a continious number, normalization helped a bit

0.3426179883351269


In [ ]:
# get Hopsworks Model Registry handle
mr = project.get_model_registry()

# Save model locally
import joblib
from pathlib import Path
modelpath = Path("tmp")/f"weather_prediction_model_{fg.version}.joblib"
joblib.dump(model, modelpath)



# Model evaluation metrics
metrics = {"accuracy": accuracy}

hs_model = mr.sklearn.create_model("weather_prediction_model", metrics=metrics, version=fg.version, description="Joblib model file", feature_view=fv)

hs_model.save(modelpath)

Uploading /home/tobias/programmieren/aiops_mlops_project/tmp/weather_prediction_model_2.joblib: 100.000%|██████████| 5968593/5968593 elapsed<00:02 remaining<00:00
Uploading /home/tobias/programmieren/aiops_mlops_project/model_schema.json: 100.000%|██████████| 1191/1191 elapsed<00:01 remaining<00:00
Model export complete: 100%|██████████| 6/6 [00:11<00:00,  1.94s/it]                   

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/31926/models/weather_prediction_model/2


Model(name: 'weather_prediction_model', version: 2)